In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.005  # Very small noise

SHRINK_TOWARD_MEAN = 0.5  # How much to shrink predictions toward historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order, excluding targets
original_order = [col for col in df.columns if col not in TARGET_COLUMNS]

# Determine dynamic features
excluded_cols = set(STATIC_FEATURES + TARGET_COLUMNS + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip features with no variance

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        # Comment out biannual if needed
        # model.add_seasonality(name='biannual', period=182.5, fourier_order=3)

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Minimal noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink toward historical mean
        historical_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * historical_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Hard clipping: stay within 5th–90th percentile
        lower_bound = ts['y'].quantile(0.05)
        upper_bound = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=lower_bound, upper=upper_bound)

        # Optional: Smooth results
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder columns to match original file (excluding targets)
    final_columns = [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[['date'] + final_columns]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_features_small_conservative.csv", index=False)
print("✅ Conservative synthetic dataset saved as 'synthetic_features_small_conservative.csv'")


### Full Synthesis

In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.01  # Small noise
SHRINK_TOWARD_MEAN = 0.3  # Gently shrink to historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order
original_order = df.columns.tolist()

# Dynamic features now includes both predictors + target columns
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip constant features

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small Gaussian noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink forecast gently toward historical mean
        hist_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * hist_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Clip to realistic bounds (5th–90th percentile)
        q5 = ts['y'].quantile(0.05)
        q90 = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=q5, upper=q90)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder to match original file (now includes targets)
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_all_features_incl_targets.csv", index=False)
print("✅ Full synthetic dataset with targets saved as 'synthetic_all_features_incl_targets.csv'")


In [ ]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np
from sklearn.utils import resample

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]
DEFAULT_NOISE = 0.01  # Small noise for synthetic data
SHRINK_TOWARD_MEAN = 0.3  # Shrink predictions toward historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3
DOWNSAMPLE_FREQ = '3M'  # Downsample to quarterly data
MAX_SAMPLES_PER_CITY = 100  # Cap samples per city for balancing

# === LOAD DATA === #
try:
    df = pd.read_csv(FILENAME)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    raise

# Ensure all expected columns exist
missing_cols = [col for col in STATIC_FEATURES + TARGET_COLUMNS + ['date'] if col not in df.columns]
if missing_cols:
    print(f"❌ Missing columns in dataset: {missing_cols}")
    raise ValueError("Dataset missing required columns")

# Save original column order
original_order = df.columns.tolist()

# Dynamic features include predictors + target columns
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# === DIAGNOSTICS FOR KAMPALA === #
kampala_df = df[df['city'] == 'Kampala']
if not kampala_df.empty:
    print("Kampala Data Summary:")
    print(kampala_df[TARGET_COLUMNS].describe())
    for col in TARGET_COLUMNS:
        if col in kampala_df.columns:
            print(f"Kampala {col} unique values: {kampala_df[col].nunique()}")
            q1, q3 = kampala_df[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            outliers = ((kampala_df[col] < q1 - 1.5 * iqr) | (kampala_df[col] > q3 + 1.5 * iqr)).sum()
            print(f"Kampala {col} outliers (beyond 1.5*IQR): {outliers}")
else:
    print("⚠️ No data for Kampala found")

# === DOWNSAMPLING AND BALANCING FUNCTION === #
def downsample_and_balance(city_df, freq=DOWNSAMPLE_FREQ, max_samples=MAX_SAMPLES_PER_CITY):
    try:
        # Ensure date is index for resampling
        city_df = city_df.set_index('date')
        # Resample to specified frequency, aggregating numeric and static columns
        agg_dict = {col: 'mean' for col in dynamic_features if col in city_df.columns}
        agg_dict.update({col: 'first' for col in STATIC_FEATURES if col in city_df.columns})
        city_df = city_df.resample(freq).agg(agg_dict).reset_index()

        # Handle missing values after resampling
        city_df = city_df.fillna(method='ffill').fillna(method='bfill')

        # Cap samples to balance dataset
        if len(city_df) > max_samples:
            city_df = resample(city_df, n_samples=max_samples, random_state=42)

        # Stratified sampling for target columns
        for target in TARGET_COLUMNS:
            if target in city_df.columns and city_df[target].nunique() > 1:
                try:
                    bins = pd.qcut(city_df[target], q=4, duplicates='drop', labels=False)
                    city_df = city_df.groupby(bins, group_keys=False).apply(
                        lambda x: resample(x, n_samples=min(len(x), max_samples // 4), random_state=42)
                    ).reset_index(drop=True)
                except Exception as e:
                    print(f"⚠️ Failed to stratify {target} for city: {e}")
                    continue

        return city_df
    except Exception as e:
        print(f"❌ Error in downsample_and_balance: {e}")
        return city_df

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    
    # Downsample and balance historical data
    city_df = downsample_and_balance(city_df)
    if city_df.empty:
        print(f"⚠️ No data after downsampling for {city}")
        continue
    
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        if feature not in city_df.columns:
            continue
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2 or ts['y'].isna().all():
            print(f"⚠️ Skipping {feature} for {city}: insufficient unique values or all NaN")
            continue

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        # Create future dates with downsampled frequency
        future = model.make_future_dataframe(periods=FUTURE_PERIODS // 3, freq=DOWNSAMPLE_FREQ)
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS // 3).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small Gaussian noise
        std_dev = ts['y'].std()
        if not np.isnan(std_dev):
            noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
            predicted[feature] += noise

        # Shrink forecast toward historical mean
        hist_mean = ts['y'].mean()
        if not np.isnan(hist_mean):
            predicted[feature] = (
                SHRINK_TOWARD_MEAN * hist_mean +
                (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
            )

        # Clip to realistic bounds (5th–90th percentile)
        q5 = ts['y'].quantile(0.05)
        q90 = ts['y'].quantile(0.90)
        if not (np.isnan(q5) or np.isnan(q90)):
            predicted[feature] = predicted[feature].clip(lower=q5, upper=q90)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date', how='outer')

    # Handle missing values after merging
    forecasted_features = forecasted_features.fillna(method='ffill').fillna(method='bfill')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly
    forecasted_features['city'] = city

    # Reorder to match original file
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
if synthetic_data_all:
    final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

    # Balance final dataset across cities
    city_counts = final_synthetic_df['city'].value_counts()
    min_samples = city_counts.min()
    balanced_df = pd.concat([
        resample(final_synthetic_df[final_synthetic_df['city'] == city], n_samples=min_samples, random_state=42)
        for city in final_synthetic_df['city'].unique()
    ], ignore_index=True)

    # Save output
    balanced_df.to_csv("datasets/synthetic_downsampled_balanced.csv", index=False)
    print("✅ Downsampled and balanced synthetic dataset saved as 'synthetic_downsampled_balanced.csv'")
else:
    print("❌ No synthetic data generated")

# Final diagnostics
if 'balanced_df' in locals():
    print("Final Dataset Summary:")
    print(balanced_df['city'].value_counts())
    print(balanced_df[TARGET_COLUMNS].describe())

### Sarima Forecasting.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
csv_file = 'datasets/engineered_dataset_with_targets.csv'  # Replace with your file path
try:
    df = pd.read_csv(csv_file, skipinitialspace=True, encoding='utf-8')
except Exception as e:
    print(f"Error loading CSV: {e}")
    raise

# Verify columns
required_columns = ['date', 'city', 'monsoon_intensity', 'climate_change', 'siltation',
                   'landslide_risks', 'rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation',
                   'high_vegetation_cover', 'geopotential_height', 'soil_volume_water_content_level1',
                   'soil_volume_water_content_level2', 'soil_volume_water_content_level3',
                   'soil_volume_water_content_level4']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}. Available columns: {df.columns.tolist()}")

# Convert date to datetime (YYYY-MM-DD)
try:
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')
except Exception as e:
    print(f"Error converting 'date' to datetime: {e}")
    raise

# Handle invalid dates
if df['date'].isna().any():
    print(f"Warning: {df['date'].isna().sum()} invalid dates found. Dropping rows.")
    df = df.dropna(subset=['date'])

# Aggregate to monthly frequency
df['month_period'] = df['date'].dt.to_period('M')
df = df.groupby(['city', 'month_period']).first().reset_index()
df['date'] = df['month_period'].dt.to_timestamp()
df = df.drop(columns=['month_period'])

# Sort and set index
df = df.sort_values(['city', 'date']).set_index('date')

# Check data sufficiency
cities = df['city'].unique()
for city in cities:
    city_df = df[df['city'] == city]
    if len(city_df) < 12:
        print(f"Warning: {city} has only {len(city_df)} months of data. Skipping (need at least 12).")
        df = df[df['city'] != city]

if df.empty:
    raise ValueError("No cities with sufficient data (at least 12 months).")

# Print data summary
print("\nData Summary:")
print(df.groupby('city').size())
print("\nMissing Values:")
print(df[required_columns[1:]].isna().sum())  # Exclude 'date' since it's the index

# Define targets and exogenous features
targets = ['monsoon_intensity', 'climate_change', 'siltation', 'landslide_risks']
exog_features = ['rainfall_mm', 'temperature_2m', 'runoff', 'total_precipitation', 'high_vegetation_cover',
                 'geopotential_height', 'soil_volume_water_content_level1', 'soil_volume_water_content_level2',
                 'soil_volume_water_content_level3', 'soil_volume_water_content_level4']

# Create future date range (2025-07 to 2050-12)
future_dates = pd.date_range(start='2025-07-01', end='2050-12-01', freq='MS')
n_future = len(future_dates)

# Initialize results
forecasts = []
mae_scores = {}
r2_scores = {}

# Function to forecast exogenous features using SARIMA
def forecast_exog(series, n_future, seasonal_period=12):
    try:
        if len(series) < 12:
            print(f"Insufficient data for exogenous series ({len(series)} months). Using mean.")
            return np.full(n_future, series.mean())
        model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(1, 1, 1, seasonal_period),
                        enforce_stationarity=False, enforce_invertibility=False)
        fitted_model = model.fit(disp=False)
        forecast = fitted_model.forecast(steps=n_future)
        return forecast
    except Exception as e:
        print(f"Error forecasting exogenous series: {e}")
        return np.full(n_future, series.mean())  # Fallback to mean

# Train SARIMAX for each city and target
for city in cities:
    print(f"\nTraining models for city: {city}")
    city_df = df[df['city'] == city].copy()
    
    # Check data sufficiency
    if len(city_df) < 12:
        print(f"Skipping {city}: only {len(city_df)} months of data.")
        continue
    
    # Forecast exogenous features for 2025-07 to 2050-12
    exog_future = pd.DataFrame(index=future_dates)
    for exog in exog_features:
        if exog.startswith('soil_volume_water_content') or exog == 'geopotential_height':
            exog_future[exog] = city_df[exog].iloc[-1]  # Use last known value
        else:
            exog_future[exog] = forecast_exog(city_df[exog], n_future)
    
    for target in targets:
        print(f"  Processing {target}")
        # Prepare endogenous and exogenous data
        endog = city_df[target]
        exog = city_df[exog_features].fillna(city_df[exog_features].mean())
        
        # Split into train (80%) and test (20%)
        train_size = int(len(endog) * 0.8)
        if train_size < 12:
            print(f"  Skipping {target}: insufficient training data ({train_size} months).")
            continue
        train_endog = endog[:train_size]
        train_exog = exog[:train_size]
        test_endog = endog[train_size:]
        test_exog = exog[train_size:]
        
        # Train SARIMAX
        try:
            model = SARIMAX(
                train_endog,
                exog=train_exog,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            fitted_model = model.fit(disp=False)
            
            # Forecast for test period (for MAE only)
            forecast_test = fitted_model.forecast(steps=len(test_endog), exog=test_exog)
            max_value = 17 if target == 'climate_change' else 16
            forecast_test = np.round(forecast_test).astype(int).clip(0, max_value)
            
            # Calculate MAE
            if len(test_endog) > 0:
                mae = mean_absolute_error(test_endog, forecast_test)
                mae_scores[f"{city}_{target}"] = mae
                print(f"  {target} Test MAE: {mae:.2f}")
            else:
                print(f"  No test data for {target}. Skipping MAE.")

            # Calculate R²
            if len(test_endog) > 0:
                r2 = r2_score(test_endog, forecast_test)
                r2_scores[f"{city}_{target}"] = r2
                print(f"  {target} Test R²: {r2:.2f}")
            else:
                print(f"  No test data for {target}. Skipping R2.")
            
            # Forecast for future period
            forecast_future = fitted_model.forecast(steps=n_future, exog=exog_future)
            forecast_future = np.round(forecast_future).astype(int).clip(0, max_value)
            
            # Store future forecasts only
            forecast_df = pd.DataFrame({
                'date': future_dates,
                'city': city,
                'target': target,
                'forecast': forecast_future
            })
            forecasts.append(forecast_df)
            
        except Exception as e:
            print(f"  Error fitting SARIMAX for {target} in {city}: {e}")
            continue

# Check if forecasts were generated
if not forecasts:
    raise ValueError("No forecasts generated. Check data or model parameters.")

# Combine and pivot forecasts
forecast_df = pd.concat(forecasts, ignore_index=True)
forecast_pivot = forecast_df.pivot_table(
    values='forecast',
    index=['date', 'city'],
    columns='target',
    aggfunc='first'
).reset_index()

# Ensure all target columns are present
for target in targets:
    if target not in forecast_pivot.columns:
        forecast_pivot[target] = np.nan

# Reorder columns
output_columns = ['date', 'city'] + targets
forecast_pivot = forecast_pivot[output_columns]

# Save to CSV
forecast_pivot.to_csv('sarimax_forecasts_2025_2050.csv', index=False)

# Print MAE scores
print("\nTest Mean Absolute Error and R2 Scores:")
for key, value in mae_scores.items():
    print(f"{key}: {value:.2f}")
for key, value in r2_scores.items():
    print(f"{key}: {value:.2f}")
print("\nFuture forecasts (2025-07 to 2050-12) saved to 'sarimax_forecasts_2025_2050.csv'")

ValueError: Missing columns: ['agricultural_practices']. Available columns: ['city', 'date', 'geopotential_height', 'high_vegetation_cover', 'high_vegetation_type', 'lake_cover', 'land_sea_mask', 'low_vegetation_cover', 'low_vegetation_type', 'soil_type', 'target_latitude', 'target_longitude', 'grid_latitude', 'grid_longitude', 'total_precipitation', 'runoff', 'evaporation', 'dewpoint_temperature_2m', 'experiment_version', 'skin_reservoir_content', 'skin_temperature', 'soil_temperature_level1', 'soil_temperature_level2', 'soil_temperature_level3', 'soil_temperature_level4', 'soil_volume_water_content_level1', 'soil_volume_water_content_level2', 'soil_volume_water_content_level3', 'soil_volume_water_content_level4', 'surface_pressure', 'temperature_2m', 'u_component_wind_10m', 'v_component_wind_10m', 'rainfall_mm', 'rainfall_monthly_anomaly', 'rainfall_3month_avg', 'rainfall_6month_avg', 'monsoon_intensity', 'climate_change', 'siltation', 'landslide_risks']